# Death Simplices & SVI Choropleth — LA County Fire Station Coverage
## MATH 497 Research — TDA

Reproduces the analysis from **O'Neil & Tymochko (2024) — Cooling Centers** applied to LA County:

| Figure | Content |
|---|---|
| **Figure A** | H1 death simplices (triangles) colored by death time |
| **Figure B** | SVI choropleth (amt. above avg.) + top H0 / H1 death simplices |

**Point cloud:** 412 fire stations → Vietoris-Rips filtration (Euclidean distance, UTM metres)  
**Base map:** 272 LA County neighborhoods with CDC SVI percentile rankings (`rpl_themes`)

## Step 1 — Dependencies

In [ ]:
# Run once; restart kernel if installing for the first time
# %pip install gudhi geopandas pyproj shapely scipy matplotlib pandas

## Step 2 — Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import geopandas as gpd
import gudhi
from shapely.geometry import Point
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.collections import PolyCollection, LineCollection
import pandas as pd

# ── Paths ──
STATIONS_PATH  = "../../data/all_stations.geojson"
SVI_PATH       = "../../data/la_county_svi.geojson"   # 272 neighborhoods + SVI
LA_COUNTY_PATH = "../../data/la_county.geojson"       # county boundary

# ── CRS ──
CRS_WGS84 = "EPSG:4326"
CRS_UTM   = "EPSG:32611"   # UTM Zone 11N — LA region

# ── Rips filtration ──
RIPS_MAX_M = 25_000   # 25 km maximum edge length

# ── How many top death simplices to highlight in Figure B ──
TOP_N_H0 = 15   # top H0 death edges by persistence
TOP_N_H1 = 5    # top H1 death triangles by persistence

print("Imports OK.")

## Step 3 — Load & Project Data

In [ ]:
# ── Fire stations ──
stations_wgs = gpd.read_file(STATIONS_PATH).set_crs(CRS_WGS84, allow_override=True)
print(f"Stations loaded  : {len(stations_wgs)}")

# ── SVI neighborhoods ──
svi_wgs = gpd.read_file(SVI_PATH).set_crs(CRS_WGS84, allow_override=True)
print(f"Neighborhoods    : {len(svi_wgs)}")
print(f"SVI columns      : rpl_themes range [{svi_wgs['rpl_themes'].min():.3f}, {svi_wgs['rpl_themes'].max():.3f}]")

# ── LA County boundary (for clipping) ──
county_wgs = gpd.read_file(LA_COUNTY_PATH).set_crs(CRS_WGS84, allow_override=True)

# ── Project to UTM Zone 11N ──
stations_proj = stations_wgs.to_crs(CRS_UTM)
svi_proj      = svi_wgs.to_crs(CRS_UTM)
county_proj   = county_wgs.to_crs(CRS_UTM)

# ── Clip stations to county boundary ──
county_union = unary_union(county_proj.geometry)
stations_proj = stations_proj[stations_proj.geometry.within(county_union)].reset_index(drop=True)

print(f"Stations (clipped): {len(stations_proj)}")

# ── Point cloud: (N, 2) array of station UTM coordinates ──
coords_m = np.column_stack([stations_proj.geometry.x, stations_proj.geometry.y])
print(f"Point cloud shape : {coords_m.shape}")

# ── SVI: compute 'amount above average rpl_themes' ──
svi_mean = svi_proj["rpl_themes"].mean()
svi_proj = svi_proj.copy()
svi_proj["svi_above_avg"] = svi_proj["rpl_themes"] - svi_mean
print(f"SVI mean (rpl_themes) : {svi_mean:.4f}")
print(f"svi_above_avg range   : [{svi_proj['svi_above_avg'].min():.3f}, {svi_proj['svi_above_avg'].max():.3f}]")

## Step 4 — Vietoris-Rips Persistent Homology

Build the Rips filtration on fire station locations. As ε grows:
- **H0** bars → when two isolated clusters merge (edge = death simplex)
- **H1** bars → when a loop of stations fills in (triangle = death simplex)

In [ ]:
print(f"Building Rips complex: N={len(coords_m)}, max_ε={RIPS_MAX_M/1000:.0f} km ...")

rips    = gudhi.RipsComplex(points=coords_m, max_edge_length=RIPS_MAX_M)
st      = rips.create_simplex_tree(max_dimension=2)

print(f"  Simplices : {st.num_simplices():,}")
print(f"  Vertices  : {st.num_vertices():,}")

st.compute_persistence(homology_coeff_field=2, min_persistence=0)
print("Persistence computed.")

# ── Quick summary ──
H0_pairs, H1_pairs = [], []
for dim, (b, d) in st.persistence():
    if np.isinf(d):
        continue
    if dim == 0:
        H0_pairs.append((b, d))
    elif dim == 1:
        H1_pairs.append((b, d))

print(f"H0 finite pairs : {len(H0_pairs)}")
print(f"H1 finite pairs : {len(H1_pairs)}")

## Step 5 — Extract Death Simplices

`st.persistence_pairs()` returns `(birth_simplex, death_simplex)` as vertex-index lists:
- **H0**: birth = `[v]`, death = edge `[u, w]`  
- **H1**: birth = edge `[u, v]`, death = triangle `[u, v, w]`

We record the coordinates and filtration (death) value of each death simplex.

In [ ]:
h0_deaths = []   # {birth_val, death_val, pers, verts, coords (2,2)}
h1_deaths = []   # {birth_val, death_val, pers, verts, coords (3,2)}

for birth_simp, death_simp in st.persistence_pairs():
    if len(death_simp) == 0:           # essential (infinite) — skip
        continue

    b_val = st.filtration(birth_simp)
    d_val = st.filtration(death_simp)
    pers  = d_val - b_val
    dim   = len(birth_simp) - 1        # 0 → H0, 1 → H1

    if dim == 0 and len(death_simp) == 2:    # H0 death: edge
        h0_deaths.append({
            "birth_val" : b_val,
            "death_val" : d_val,
            "pers"      : pers,
            "verts"     : death_simp,
            "coords"    : coords_m[death_simp],    # (2, 2)
        })

    elif dim == 1 and len(death_simp) == 3:  # H1 death: triangle
        h1_deaths.append({
            "birth_val" : b_val,
            "death_val" : d_val,
            "pers"      : pers,
            "verts"     : death_simp,
            "coords"    : coords_m[death_simp],    # (3, 2)
        })

# Sort by persistence (descending)
h0_deaths.sort(key=lambda x: -x["pers"])
h1_deaths.sort(key=lambda x: -x["pers"])

print(f"H0 death simplices (edges)     : {len(h0_deaths)}")
print(f"H1 death simplices (triangles) : {len(h1_deaths)}")

if h1_deaths:
    print(f"\nTop 5 H1 death simplices by persistence:")
    print(f"{'Rank':>4}  {'Birth (km)':>10}  {'Death (km)':>10}  {'Pers (km)':>10}")
    for i, d in enumerate(h1_deaths[:5], 1):
        print(f"{i:>4}  {d['birth_val']/1000:>10.2f}  {d['death_val']/1000:>10.2f}  {d['pers']/1000:>10.2f}")

## Figure A — H1 Death Simplices Colored by Death Time

Analogous to **Figure 8** in O'Neil & Tymochko (2024).  
Each triangle is the 2-simplex that killed a 1-dimensional homology class (coverage loop).  
Color encodes the **death time** in kilometres — yellow = largest death value (loop persisted longest).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))

# ── Base map: neighborhood boundaries (light grey) ──
svi_proj.boundary.plot(ax=ax, color="#999999", linewidth=0.4, alpha=0.6)

# ── H1 death triangles, colored by death time ──
if h1_deaths:
    death_vals_km = np.array([d["death_val"] / 1000 for d in h1_deaths])
    vmin, vmax    = death_vals_km.min(), death_vals_km.max()
    cmap          = cm.plasma
    norm          = mcolors.Normalize(vmin=vmin, vmax=vmax)

    tri_verts  = [d["coords"] for d in h1_deaths]   # list of (3,2) arrays
    tri_colors = [cmap(norm(v)) for v in death_vals_km]

    poly_coll = PolyCollection(
        tri_verts,
        facecolors=tri_colors,
        edgecolors="none",
        alpha=0.75,
        zorder=3,
    )
    ax.add_collection(poly_coll)

    # Colorbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.45, pad=0.02)
    cbar.set_label("Death Time (kilometres)", fontsize=11)

# ── Fire stations (small black dots) ──
ax.scatter(
    stations_proj.geometry.x, stations_proj.geometry.y,
    c="black", s=6, alpha=0.6, zorder=5, label="Fire stations"
)

ax.set_title(
    "Death Simplices for the 1-Dimensional Homology Classes\n"
    "LA County — Fire Station Vietoris-Rips Filtration",
    fontsize=13,
)
ax.legend(fontsize=9, loc="lower right")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("../../results/death_simplices_H1_LA.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/death_simplices_H1_LA.png")

## Figure B — SVI Choropleth + Top Death Simplices

Analogous to **Figure 9** in O'Neil & Tymochko (2024).  

- **Background**: each neighborhood is shaded by **amount above average SVI** (`rpl_themes − mean`).  
  Pink = above-average vulnerability; green = below-average.
- **Dark blue lines**: top H0 death simplices (edges) — where isolated clusters last merged  
- **Blue triangles**: top H1 death simplices (triangles) — the most persistent coverage voids

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))

# ── Choropleth: SVI amount above average ──
abs_max   = svi_proj["svi_above_avg"].abs().max()
svi_proj.plot(
    column     = "svi_above_avg",
    ax         = ax,
    cmap       = "PiYG_r",          # pink = high SVI, green = low SVI
    vmin       = -abs_max,
    vmax       =  abs_max,
    legend     = True,
    alpha      = 0.75,
    legend_kwds= {
        "label"  : "Amt. Above Avg. SVI (rpl_themes)",
        "shrink" : 0.45,
        "pad"    : 0.02,
    },
)

# Neighborhood boundaries
svi_proj.boundary.plot(ax=ax, color="white", linewidth=0.3, alpha=0.5)

# ── Top H0 death edges (dark blue lines) ──
top_h0 = h0_deaths[:TOP_N_H0]
if top_h0:
    segs = [d["coords"] for d in top_h0]   # list of (2,2) arrays
    lc   = LineCollection(
        segs,
        colors     = "#1a237e",   # dark navy
        linewidths = 2.0,
        zorder     = 4,
        label      = f"Top H0 death simplex (Dim 0, n={len(top_h0)})",
    )
    ax.add_collection(lc)

# ── Top H1 death triangles (translucent blue) ──
top_h1 = h1_deaths[:TOP_N_H1]
if top_h1:
    tri_verts = [d["coords"] for d in top_h1]
    poly_coll = PolyCollection(
        tri_verts,
        facecolors = "#5c6bc0",   # medium blue
        edgecolors = "#1a237e",
        linewidths = 1.0,
        alpha      = 0.65,
        zorder     = 5,
        label      = f"Top H1 death simplex (Dim 1, n={len(top_h1)})",
    )
    ax.add_collection(poly_coll)

# ── Legend patches (mimicking paper style) ──
above_patch = mpatches.Patch(color="#c51b7d", alpha=0.75, label="Tract w/ ABOVE Avg. SVI")
below_patch = mpatches.Patch(color="#4d9221", alpha=0.75, label="Tract w/ BELOW Avg. SVI")
h0_line     = plt.Line2D([0], [0], color="#1a237e", linewidth=2.5,
                          label=f"Top Death Simplex (Dim 0, top {TOP_N_H0})")
h1_patch    = mpatches.Patch(color="#5c6bc0", alpha=0.65,
                              label=f"Top Death Simplex (Dim 1, top {TOP_N_H1})")

ax.legend(
    handles   = [above_patch, below_patch, h0_line, h1_patch],
    loc       = "lower left",
    fontsize  = 9,
    framealpha= 0.9,
    ncol      = 2,
)

ax.set_title(
    "SVI Vulnerability & Top Persistent Homology Death Simplices\n"
    "LA County — Fire Station Coverage Analysis",
    fontsize=13,
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("../../results/svi_choropleth_death_simplices_LA.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/svi_choropleth_death_simplices_LA.png")

## Figure C — Combined: Death Simplices by Dimension (Side-by-Side)

Shows **all** H0 and H1 death simplices colored by death time, side by side — useful for comparing where components merged vs. where loops filled in.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for ax, deaths, dim_label, cmap_name in [
    (axes[0], h0_deaths, "Dim 0 — H0 Death Edges\n(cluster merges)",       "viridis"),
    (axes[1], h1_deaths, "Dim 1 — H1 Death Triangles\n(loop fill-ins)",   "plasma"),
]:
    # Base map
    svi_proj.boundary.plot(ax=ax, color="#aaaaaa", linewidth=0.35, alpha=0.6)

    if deaths:
        death_vals_km = np.array([d["death_val"] / 1000 for d in deaths])
        norm  = mcolors.Normalize(vmin=death_vals_km.min(), vmax=death_vals_km.max())
        cmap  = cm.get_cmap(cmap_name)

        if dim_label.startswith("Dim 0"):     # edges
            segs   = [d["coords"] for d in deaths]
            colors = [cmap(norm(v)) for v in death_vals_km]
            lc     = LineCollection(segs, colors=colors, linewidths=1.2, alpha=0.7, zorder=3)
            ax.add_collection(lc)
        else:                                  # triangles
            tri_verts  = [d["coords"] for d in deaths]
            tri_colors = [cmap(norm(v)) for v in death_vals_km]
            poly_coll  = PolyCollection(
                tri_verts, facecolors=tri_colors, edgecolors="none", alpha=0.75, zorder=3
            )
            ax.add_collection(poly_coll)

        sm   = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, shrink=0.45, pad=0.02)
        cbar.set_label("Death Time (kilometres)", fontsize=10)

    # Fire stations
    ax.scatter(
        stations_proj.geometry.x, stations_proj.geometry.y,
        c="black", s=5, alpha=0.55, zorder=5
    )
    ax.set_title(f"LA County — {dim_label}", fontsize=11)
    ax.set_axis_off()

plt.suptitle(
    "Death Simplices by Homological Dimension\nFire Station Vietoris-Rips — LA County",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig("../../results/death_simplices_dim0_dim1_LA.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/death_simplices_dim0_dim1_LA.png")

## Figure D — Persistence Diagram

Birth vs. death scatter for all finite H0 and H1 pairs. Features far from the diagonal represent the most structurally significant coverage gaps. The death simplices plotted above correspond to these points.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for pairs, color, label in [
    (H0_pairs, "steelblue", "H0 (components)"),
    (H1_pairs, "tomato",    "H1 (loops / voids)"),
]:
    if not pairs:
        continue
    arr  = np.array(pairs) / 1000   # metres → km
    b, d = arr[:, 0], arr[:, 1]
    pers = d - b
    sizes = np.clip(pers / pers.max() * 120, 8, 120) if pers.max() > 0 else np.full(len(pers), 10)
    ax.scatter(b, d, c=color, s=sizes, alpha=0.65, edgecolors="k", linewidths=0.3, label=label)

all_vals = [v for p in H0_pairs + H1_pairs for v in p]
lim = max(all_vals) / 1000 * 1.05 if all_vals else 1
ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.6)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_aspect("equal")
ax.set_xlabel("Birth (km)", fontsize=11)
ax.set_ylabel("Death (km)", fontsize=11)
ax.set_title("Persistence Diagram — Vietoris-Rips\nFire Stations, LA County", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("../../results/persistence_diagram_LA.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/persistence_diagram_LA.png")

## Step 6 — Tabular Summary of Top Death Simplices

In [ ]:
def simplex_summary(deaths, dim, n=10):
    rows = []
    for i, d in enumerate(deaths[:n], 1):
        # Find which SVI neighborhood each vertex belongs to
        verts = d["verts"]
        hoods = []
        for v in verts:
            pt = Point(coords_m[v])
            name = "—"
            for _, row in svi_proj.iterrows():
                if row.geometry.contains(pt):
                    name = row["name"]
                    break
            hoods.append(name)
        rows.append({
            "rank"         : i,
            "dim"          : dim,
            "birth_km"     : round(d["birth_val"] / 1000, 2),
            "death_km"     : round(d["death_val"] / 1000, 2),
            "pers_km"      : round(d["pers"] / 1000, 2),
            "vertex_hoods" : " | ".join(hoods),
        })
    return pd.DataFrame(rows)

print("=== Top H0 Death Edges ===")
df_h0 = simplex_summary(h0_deaths, 0, n=10)
display(df_h0)

print("\n=== Top H1 Death Triangles ===")
df_h1 = simplex_summary(h1_deaths, 1, n=10)
display(df_h1)